# Day 27 · Trace every step with OpenTelemetry

**First-principles question:** When a reply is wrong, which step made it wrong?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import json
import pandas as pd
from lab import llm
from lab.acme import load_cases, save_results
from lab.agent import SYSTEM_C, run_agent
from lab.evals import score
from lab.tracing import TRACES, model_span, provenance, tool_span, tracer

cases = load_cases()

## 1 · Run C with a span per step
`run_agent` already returns a trace; turn each step into child spans.

In [ ]:
(TRACES / "spans.jsonl").unlink(missing_ok=True)
rows = []
for case in cases:
    with tracer.start_as_current_span("invoke_agent", attributes={"gen_ai.operation.name": "invoke_agent",
                                                                   "acme.case_id": case["id"]}):
        out, steps = run_agent(case)
        for step in steps:
            with tracer.start_as_current_span("chat", attributes=model_span(llm.CHAT_MODEL, step)):
                for name, args, _ in step.get("tool_calls", []):
                    with tracer.start_as_current_span(f"execute_tool {name}", attributes=tool_span(name, args)):
                        pass
    rows.append(dict(out, id=case["id"], steps=len(steps), tools=[c[0] for s in steps for c in s.get("tool_calls", [])]))
meta = provenance(llm.CHAT_MODEL, SYSTEM_C)
save_results("C-traced", [dict(r, **meta) for r in rows])
print(meta)

## 2 · Trace graders

In [ ]:
def lookup_first(row):
    return bool(row["tools"]) and row["tools"][0] == "lookup_customer"

def no_repeats(row):
    # TODO: True when no tool name appears twice in a row
    ...

def escalates_when_needed(row, case):
    # TODO: if case["needs_human"], "escalate" must be among row["tools"]
    ...

def short(row):
    return row["steps"] <= 6

by_id = {c["id"]: c for c in cases}
table = pd.DataFrame([{"id": r["id"], "answer_pass": score(by_id[r["id"]], r)["passed"],
                       "trace_pass": all([lookup_first(r), no_repeats(r), escalates_when_needed(r, by_id[r["id"]]), short(r)])}
                      for r in rows])
print(table[["answer_pass", "trace_pass"]].value_counts())
table[table.answer_pass & ~table.trace_pass]

**A case that passed for the wrong reason, and what the trace shows:** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert set(meta) == {"model_id", "prompt_sha", "grader_sha", "dataset_sha"}, "provenance(): four keys"
spans = [json.loads(l) for l in (TRACES / "spans.jsonl").read_text().splitlines()]
assert any(s["attributes"].get("gen_ai.usage.input_tokens") is not None for s in spans), "model_span(): token attributes"
assert no_repeats({"tools": ["a", "b", "a"]}) and not no_repeats({"tools": ["a", "a"]})
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")